# Install Necessay Libraries

In [1]:
!pip install -qU langchain langchain-text-splitters langchain-cohere langchain-qdrant langchain-community
!pip install -qU qdrant-client beautifulsoup4 lxml
!pip install -qU pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━

# Import Libraries


In [2]:
import os #read API
import cohere

from google.colab import userdata
from langchain_community.document_loaders import WebBaseLoader , PyPDFLoader # Fetches and parses web pages into Documents
from langchain_text_splitters import RecursiveCharacterTextSplitter #Splits long Documents into smaller chunks
from langchain_cohere import CohereEmbeddings,ChatCohere
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

/tmp/ipykernel_1459/2196447858.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader , PyPDFLoader # Fetches and parses web pages into Documents


# Config

In [3]:
#Load Secrets
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
print('Api Key Loaded!')

#configuration
cohere_embed_model = 'embed-english-v3.0'
cohere_chat_model = 'command-r-08-2024'
cohere_rerank_model = 'rerank-v4.0-pro'
qdrant_collection = 'RAG_web-pdf'

#chunks the sources
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 4
RETRIEVE_K = 15    # wide retrieval before reranking
print('Configuration Set!')

Api Key Loaded!
Configuration Set!


# Load Document

In [4]:
print('LOADING & PROCESSING DOCUMENT')
print('='*60)

urls = [
    'https://docs.cohere.com/docs/embeddings',
    'https://docs.langchain.com/oss/python/langchain/retrieval'

    ]


print('Loading Document')
load_url = WebBaseLoader(web_paths=urls)
load_pdf = PyPDFLoader('/content/Chapter_4_v9.0[1].pdf')

load_docs = load_url.load() + load_pdf.load()
print(f'Loaded {len(load_docs)} documents')

print('Split into Chunks')
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = text_spliter.split_documents(load_docs)
print(f'Split into {len(splits)} chunks')
print(f'Preview: {splits[0].page_content[:200]}')

# Check what sources were actually loaded
# sources = set([doc.metadata.get('source', 'unknown') for doc in load_docs])
# print("Sources found:")
# for source in sources:
#     print(f"  → {source}")

# web_docs = load_url.load()
# pdf_docs = load_pdf.load()
# print(f"Web documents: {len(web_docs)}")
# print(f"PDF pages: {len(pdf_docs)}")
# print(f"Total: {len(load_docs)}")

LOADING & PROCESSING DOCUMENT
Loading Document
Loaded 104 documents
Split into Chunks
Split into 147 chunks
Preview: Introduction to Embeddings at Cohere | CohereFor AI agents: a documentation index is available at the root level at /llms.txt. Append /llms.txt to any URL for a page-level index, or .md for the markdo


# Create Vector

In [ ]:
print('=' *60)
print('Creating Vector Store')

print('Initializing Embeddings')
embeddings = CohereEmbeddings(
    cohere_api_key= os.getenv('COHERE_API_KEY'),
    model = cohere_embed_model
)
#embeddings.input_type = "search_document"
test_vector = embeddings.embed_query('test')
vector_size = len(test_vector)
print(f'Vector Size: {vector_size}')

print('Connecting to Qdrant VectoreStore')
qdrant = QdrantClient(
    url=os.getenv('QDRANT_URL'),
    api_key=os.getenv('QDRANT_API_KEY')
)

existing_collection = [c.name for c in qdrant.get_collections().collections]
print(existing_collection) #empty , cluster have no collection yet

if qdrant_collection not in existing_collection:
    print('Collection not found - Creating Collection')

    qdrant.create_collection(
        collection_name = qdrant_collection,
        vectors_config= VectorParams(size=vector_size, distance=Distance.COSINE)
    )

    vectorStore = QdrantVectorStore.from_documents(
        documents = splits,
        embedding = embeddings,
        url = os.getenv('QDRANT_URL'),
        api_key = os.getenv('QDRANT_API_KEY'),
        collection_name = qdrant_collection
    )

    print(f'Stored {len(splits)} chunks!')

else:
  print('collection found - connecting')
  vectorStore = QdrantVectorStore.from_existing_collection(
      embedding = embeddings,
      collection_name = qdrant_collection,
      url = os.getenv('QDRANT_URL'),
      api_key = os.getenv('QDRANT_API_KEY')
  )
  print('connected to existing collection')




Creating Vector Store
Initializing Embeddings
Vector Size: 1024
Connecting to Qdrant VectoreStore
['RAG_web-pdf']
collection found - connecting
connected to existing collection


In [ ]:
#check url
#print(repr(os.getenv('QDRANT_URL')))

'https://a673ac83-b4f2-4ce7-9c72-a5384de9aabc.us-east-2-0.aws.cloud.qdrant.io'


# Building Rag System

In [ ]:
llm = ChatCohere(
    cohere_api_key = os.getenv('COHERE_API_KEY'),
    model = cohere_chat_model,
    temperature= 0.7,
    max_tokens = 512

)
print('='*60)

co = cohere.Client(os.getenv('COHERE_API_KEY'))

def queryRag(question: str):
  print(f'Question : {question}')
  print('='*60)

  #retrieve relevant chunks from vectorstore


  docs = vectorStore.similarity_search(
      query = question,
      k = RETRIEVE_K
  )

  re_ranked = co.rerank (
      query = question,
      documents = [doc.page_content for doc in docs],
      model = cohere_rerank_model,
      top_n = TOP_K
  )

  #join chunks into one context string

  best_docs = [docs[r.index] for r in re_ranked.results]

  documents = [
    {
        "id": f"chunk_{i}",
        "data": doc.page_content   # plain string only
    }
    for i, doc in enumerate(best_docs)
]

  # generate answer with citations
  response = co.chat(
      message = question,
      documents = documents,
      model = cohere_chat_model,
  )

  context = '\n\n'.join([doc.page_content for doc in best_docs])

  #generate answer

  print("\nANSWER:")
  print("-"*60)
  print(response.text)
  print("-"*60)

  if response.citations:
    print('\nCITATIONS:')
    for citation in response.citations:
        for doc_id in citation.document_ids:
            index = int(doc_id.split('_')[1])
            source = best_docs[index].metadata.get('source', 'unknown')
            page = best_docs[index].metadata.get('page', 'N/A')
            print(f'  · "{citation.text}"')
            print(f'    → {source}  page {page}')

  print(f"\nBased on {len(best_docs)} chunks (reranked from {len(docs)})")






In [ ]:
queryRag("What are Cohere embeddings?")

Question : What are Cohere embeddings?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
Cohere embeddings are a platform and set of models that create vector representations of various data types, including text, images, and audio. These embeddings can be used for a range of tasks such as semantic search, retrieval, and classification. The platform supports compression and allows users to specify output dimensions and embedding types.
------------------------------------------------------------
Based on 4 chunks


In [ ]:
queryRag('what is forwarding?')

Question : what is forwarding?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
Forwarding is the process of moving packets from a router's input link to the appropriate router output link. It can be likened to getting through a single interchange during a trip.
------------------------------------------------------------
Based on 4 chunks


In [ ]:
queryRag('data plane & control plane explain?')

Question : data plane & control plane explain?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
The data plane is responsible for the local, per-router function and determines how a datagram is forwarded from the router's input port to its output port. It is a fundamental part of the network layer and handles the actual data transmission. 

The control plane, on the other hand, is concerned with network-wide logic and determines how a datagram is routed among routers along the end-to-end path from the source host to the destination host. It provides the overall routing strategy and control over the network.
------------------------------------------------------------
Based on 4 chunks


In [ ]:
queryRag("How does vector search work in RAG systems?")

Question : How does vector search work in RAG systems?

ANSWER:
------------------------------------------------------------
Retrieval-Augmented Generation (RAG) is a system that combines search with generation to produce grounded, context-aware answers.

RAG works by retrieving relevant external knowledge at query time, which addresses the problems of finite context and static knowledge. This enhances an LLM's answers with context-specific information.

To build a knowledge base, you can use LangChain's document loaders and vector stores to create one from your own data. Vector stores are specialised databases for storing and searching embeddings.

An embedding model turns text into a vector of numbers so that texts with similar meaning land close together in that vector space. This allows for more accurate searches and retrieval of relevant information.
------------------------------------------------------------

CITATIONS:
  · "Retrieval-Augmented Generation"
    → https://docs.lan

In [ ]:
queryRag("What is the difference between forwarding and routing?")

Question : What is the difference between forwarding and routing?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
Forwarding is the process of moving a packet from a router's input link to the appropriate output link, much like getting through a single interchange during a trip. Routing, on the other hand, is the process of planning the entire trip from the source to the destination, determining the route taken by the packets.
------------------------------------------------------------
Based on 4 chunks (reranked from 15)


In [ ]:
queryRag("How does IPv4 addressing work?")

Question : How does IPv4 addressing work?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
IPv4 addressing works by assigning a unique 32-bit identifier, known as an IP address, to each host or router interface. These IP addresses are allocated by the Internet Corporation for Assigned Names and Numbers (ICANN) through regional registries. The 32-bit address space allows for a finite number of unique addresses, which led to concerns about exhaustion. To mitigate this, Network Address Translation (NAT) was introduced, and IPv6, with its 128-bit address space, was developed.
------------------------------------------------------------
Based on 4 chunks (reranked from 15)


In [ ]:
queryRag("How does IPv4 addressing work?")

Question : How does IPv4 addressing work?

ANSWER:
------------------------------------------------------------
IPv4 addressing uses a 32-bit identifier associated with each host or router interface. This identifier is known as an IP address.

All devices in a local network have 32-bit addresses in a "private" IP address space. These addresses can only be used within the local network and are not visible to the outside world. This means that only one IP address is needed from the provider ISP for all devices, and the addresses of hosts within the local network can be changed without notifying the outside world.

The Internet Corporation for Assigned Names and Numbers (ICANN) allocates IP addresses through five regional registries (RRs), which may then allocate to local registries. In 2011, ICANN allocated the last chunk of IPv4 addresses to the RRs.

IPv4 addresses are written in dotted-decimal notation, such as 223.1.1.1, 223.1.1.2, and so on.
-----------------------------------------

In [ ]:
queryRag("What are Cohere embeddings used for?")

Question : What are Cohere embeddings used for?

ANSWER:
------------------------------------------------------------
Cohere embeddings are used for a variety of purposes, including:

- **AI agents**: Cohere provides a documentation index for AI agents, which can be accessed at the root level at /llms.txt and /llms-full.txt.
- **Semantic search**: Cohere embeddings can be used for semantic search, which involves finding relevant information based on the meaning or context of a query.
- **Multimodal embeddings**: These are used for processing and understanding different types of data, such as text, images, and audio.
- **Batch embedding jobs**: Cohere supports batch embedding jobs, which allow for the efficient processing of large volumes of data.
- **Reranking**: Cohere embeddings can be used to rerank search results based on their relevance or importance.
- **Image embeddings**: The Cohere Embedding platform supports image embeddings for embed-v4.0 and the embed-v3.0 family.
- **Compa

In [ ]:
!pip show cohere

Name: cohere
Version: 5.21.1
Summary: 
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: fastavro, httpx, pydantic, pydantic-core, requests, tokenizers, types-requests, typing_extensions
Required-by: langchain-cohere
